# XFeat-SuperPoint Hybrid — MegaDepth Raw Training (Hardened Colab)

This notebook is designed for **Google Colab GPU runtimes** and focuses on robust setup, safe extraction, reliable downloads, and visible training progress/metrics.


In [ ]:
#@title 1) Runtime preflight (GPU + Python + Colab)
import os
import sys
import subprocess
from datetime import datetime, timezone

print('UTC now:', datetime.now(timezone.utc).isoformat(timespec='seconds'))

if 'COLAB_GPU' not in os.environ and 'google.colab' not in sys.modules:
    print('⚠️ This notebook is intended for Google Colab. Continuing anyway...')

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if smi.returncode != 0:
    raise RuntimeError('No GPU runtime detected. In Colab use Runtime → Change runtime type → GPU.')

print(smi.stdout.splitlines()[0])
print('Python:', sys.version.split()[0])
print('Executable:', sys.executable)


In [ ]:
#@title 2) Mount Google Drive
try:
    from google.colab import drive
except Exception as e:
    raise RuntimeError('google.colab is unavailable. Run this notebook in Google Colab.') from e

drive.mount('/content/drive', force_remount=False)


In [ ]:
#@title 3) Paths and training knobs
from pathlib import Path

# ===== User-configurable =====
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/hybrid_feature_matching'
RAW_ROOT_DRIVE     = f'{DRIVE_PROJECT_ROOT}/data/megadepth_raw'   # scene dirs like 0001, 0002, ...
RAW_TAR_PATH       = ''  # optional tar on Drive, e.g. '/content/drive/MyDrive/.../megadepth_raw.tar'

REPO_URL            = 'https://github.com/MalharRane/XFeat-SuperPoint-Hybrid-Model.git'
REPO_BRANCH         = ''  # optional, e.g. 'main'. Empty = repo default branch.
REPO_DIR            = '/content/XFeat-SuperPoint-Hybrid-Model'
XFEAT_REPO_URL      = 'https://github.com/verlab/accelerated_features.git'
XFEAT_DIR           = '/content/accelerated_features'
SUPERPOINT_REPO_URL = 'https://github.com/rpautrat/SuperPoint.git'
SUPERPOINT_DIR      = '/content/SuperPoint'

DRIVE_CKPT_DIR      = f'{DRIVE_PROJECT_ROOT}/checkpoints/megadepth_raw'
DRIVE_LOG_DIR       = f'{DRIVE_PROJECT_ROOT}/runs/megadepth_raw'

# Training hyperparameters
BATCH_SIZE          = 4
MAX_EPOCHS          = 50
LR                  = 1e-4
NUM_WORKERS         = 2
IMAGE_H, IMAGE_W    = 480, 640  # MUST be divisible by 8
VAL_SPLIT_RATIO     = 0.2
MAX_PAIRS_PER_SCENE = 1000
MIN_OVERLAP         = 0.15
MAX_OVERLAP         = 0.70
MIN_KEYPOINT_SCORE  = 0.01

# Resume checkpoint: leave empty to auto-pick best.pth / latest epoch_*.pth
RESUME_CKPT             = ''
NO_VERIFY_DATASET_PAIRS = False
VALIDATION_SCAN_LIMIT   = 250
OVERRIDE_CFG_PATH       = '/content/megadepth_raw_override.yaml'

Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_LOG_DIR).mkdir(parents=True, exist_ok=True)

print('REPO_DIR       :', REPO_DIR)
print('RAW_ROOT_DRIVE :', RAW_ROOT_DRIVE)
print('DRIVE_CKPT_DIR :', DRIVE_CKPT_DIR)
print('DRIVE_LOG_DIR  :', DRIVE_LOG_DIR)


In [ ]:
#@title 4) Clone/update repos + install dependencies (with retries)
import os
import sys
import time
import importlib
import subprocess
from pathlib import Path


def run_with_retry(cmd, cwd=None, env=None, max_attempts=3, sleep_s=5, capture=False):
    last = None
    for attempt in range(1, max_attempts + 1):
        try:
            print('+', ' '.join(cmd))
            return subprocess.run(cmd, cwd=cwd, env=env, check=True, text=True, capture_output=capture)
        except subprocess.CalledProcessError as e:
            last = e
            if e.stdout:
                print('stdout tail\n', e.stdout[-2000:])
            if e.stderr:
                print('stderr tail\n', e.stderr[-2000:])
            if attempt < max_attempts:
                print(f'Command failed ({attempt}/{max_attempts}), retrying in {sleep_s}s...')
                time.sleep(sleep_s)
    raise last


def ensure_repo(path, url, branch=''):
    p = Path(path)
    if p.exists() and (p / '.git').exists():
        run_with_retry(['git', '-C', str(p), 'fetch', '--all', '--tags', '--prune'], max_attempts=3)
        if branch:
            run_with_retry(['git', '-C', str(p), 'checkout', branch], max_attempts=2)
            run_with_retry(['git', '-C', str(p), 'pull', '--ff-only', 'origin', branch], max_attempts=3)
        else:
            run_with_retry(['git', '-C', str(p), 'pull', '--ff-only'], max_attempts=3)
    elif p.exists():
        raise RuntimeError(f'Path exists but is not a git repo: {p}')
    else:
        clone_cmd = ['git', 'clone', url, str(p)]
        if branch:
            clone_cmd = ['git', 'clone', '--branch', branch, '--single-branch', url, str(p)]
        run_with_retry(clone_cmd, max_attempts=3)


def pip_install(args, max_attempts=3):
    return run_with_retry([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *args], max_attempts=max_attempts)


def ensure_lightglue():
    candidates = [
        ['lightglue'],
        ['lightglue-python'],
        ['git+https://github.com/cvg/LightGlue.git'],
    ]
    last_err = None
    for cand in candidates:
        try:
            pip_install(cand, max_attempts=2)
            importlib.invalidate_caches()
            import lightglue  # noqa: F401
            print(f"LightGlue installed/importable via: {' '.join(cand)}")
            return
        except Exception as e:
            last_err = e
            print(f"LightGlue install/import failed for {' '.join(cand)}: {type(e).__name__}: {e}")
    raise RuntimeError('Could not install/import LightGlue from available sources.') from last_err


ensure_repo(REPO_DIR, REPO_URL, REPO_BRANCH.strip())
ensure_repo(XFEAT_DIR, XFEAT_REPO_URL)
ensure_repo(SUPERPOINT_DIR, SUPERPOINT_REPO_URL)

pip_install(['--upgrade', 'pip', 'setuptools', 'wheel'], max_attempts=3)
pip_install(['-r', f'{REPO_DIR}/requirements.txt'], max_attempts=3)
pip_install(['h5py', 'kornia', 'matplotlib', 'tensorboard', 'lightning-utilities'], max_attempts=3)
ensure_lightglue()

# Make repos importable in this runtime and subprocesses.
py_paths = [REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR]
existing = os.environ.get('PYTHONPATH', '')
if existing:
    py_paths.extend(existing.split(os.pathsep))
py_paths = [p for p in dict.fromkeys(py_paths) if p]
os.environ['PYTHONPATH'] = os.pathsep.join(py_paths)

for p in reversed([REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR]):
    if p not in sys.path:
        sys.path.insert(0, p)

# Quick import sanity for core libs.
for pkg in ['torch', 'numpy', 'yaml', 'h5py', 'kornia', 'matplotlib']:
    importlib.import_module(pkg)

print('Dependency setup complete.')
print('PYTHONPATH=', os.environ['PYTHONPATH'])


In [ ]:
#@title 5) Download pretrained weights (idempotent + retry)
import time
import urllib.request
from pathlib import Path

weights_dir = Path(REPO_DIR) / 'weights'
weights_dir.mkdir(parents=True, exist_ok=True)

assets = [
    ('https://github.com/verlab/accelerated_features/releases/download/v1.0/xfeat.pt', weights_dir / 'xfeat.pth'),
    ('https://github.com/magicleap/SuperPointPretrainedNetwork/raw/master/superpoint_v1.pth', weights_dir / 'superpoint_v1.pth'),
]


def download_with_retry(url, dst, attempts=3, sleep_s=4):
    last = None
    for i in range(1, attempts + 1):
        try:
            print(f'Downloading ({i}/{attempts}) {url} -> {dst}')
            urllib.request.urlretrieve(url, str(dst))
            if dst.stat().st_size <= 0:
                raise RuntimeError(f'Empty file downloaded: {dst}')
            return
        except Exception as e:
            last = e
            if i < attempts:
                print(f'Download failed: {e}; retrying in {sleep_s}s...')
                time.sleep(sleep_s)
    raise RuntimeError(f'Failed to download {url}') from last


for url, dst in assets:
    if dst.exists() and dst.stat().st_size > 0:
        print('Exists:', dst, f'({dst.stat().st_size/1e6:.1f} MB)')
        continue
    download_with_retry(url, dst)

print('Weights ready in', weights_dir)


In [ ]:
#@title 6) Optional: safely extract RAW_TAR_PATH to /content/megadepth_raw
import tarfile
from pathlib import Path


def _safe_within_directory(base: Path, target: Path) -> bool:
    base_r = base.resolve()
    target_r = target.resolve()
    try:
        target_r.relative_to(base_r)
        return True
    except ValueError:
        return False


def safe_extract(tar: tarfile.TarFile, extract_root: Path):
    for member in tar.getmembers():
        normalized = Path(member.name)
        if normalized.is_absolute() or '..' in normalized.parts:
            raise RuntimeError(f'Unsafe tar entry (invalid path): {member.name}')
        member_path = extract_root / member.name
        if not _safe_within_directory(extract_root, member_path):
            raise RuntimeError(f'Unsafe tar entry (path traversal): {member.name}')
        if member.issym() or member.islnk():
            raise RuntimeError(f'Unsafe tar entry (links not allowed): {member.name}')
    tar.extractall(extract_root)


if RAW_TAR_PATH:
    tar_path = Path(RAW_TAR_PATH)
    if not tar_path.exists():
        raise FileNotFoundError(f'RAW_TAR_PATH not found: {tar_path}')

    extract_root = Path('/content/megadepth_raw')
    extraction_flag = extract_root / '.extracted'

    if extraction_flag.exists():
        print('Already extracted:', extract_root)
    else:
        extract_root.mkdir(parents=True, exist_ok=True)
        print('Extracting', tar_path, '->', extract_root)
        with tarfile.open(tar_path, 'r:*') as tf:
            safe_extract(tf, extract_root)
        if not any(extract_root.glob('**/dense0/imgs/*')):
            raise RuntimeError('Extraction completed but no images found under **/dense0/imgs')
        extraction_flag.write_text('ok')

    RAW_ROOT = str(extract_root)
else:
    RAW_ROOT = RAW_ROOT_DRIVE

print('RAW_ROOT:', RAW_ROOT)


In [ ]:
#@title 7) Validate MegaDepth raw layout
from pathlib import Path

raw = Path(RAW_ROOT)
if not raw.exists():
    raise FileNotFoundError(f'RAW_ROOT does not exist: {raw}')

scene_dirs = sorted([p for p in raw.iterdir() if p.is_dir()])
img_dirs = sorted(raw.glob('**/dense0/imgs'))
depth_dirs = sorted(raw.glob('**/dense0/depths'))

num_imgs = 0
for d in img_dirs[:VALIDATION_SCAN_LIMIT]:
    num_imgs += len(list(d.glob('*.jpg'))) + len(list(d.glob('*.jpeg'))) + len(list(d.glob('*.png')))

num_h5 = 0
for d in depth_dirs[:VALIDATION_SCAN_LIMIT]:
    num_h5 += len(list(d.glob('*.h5')))

print('scene dirs:', len(scene_dirs))
print('img dirs  :', len(img_dirs))
print('depth dirs:', len(depth_dirs))
print('sample image count (partial scan):', num_imgs)
print('sample depth count (partial scan):', num_h5)

if len(img_dirs) == 0 or num_imgs == 0:
    raise RuntimeError('No training images found under **/dense0/imgs. Check RAW_ROOT.')

if len(depth_dirs) == 0:
    print('ℹ️ No depth folders found. Training can still run with homography-based supervision.')


In [ ]:
#@title 8) Training preflight import checks
import os
import subprocess
import sys

env = os.environ.copy()
env['PYTHONPATH'] = ':'.join([REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR, env.get('PYTHONPATH', '')]).strip(':')

test_script = r'''
import importlib
import torch

print('torch', torch.__version__, '| cuda_available=', torch.cuda.is_available())

xfeat_ok = False
for module_name, attr in [
    ('modules.xfeat', 'XFeat'),
    ('models.xfeat_core', 'XFeatCore'),
]:
    try:
        mod = importlib.import_module(module_name)
        if hasattr(mod, attr):
            print('XFeat import ok via', module_name, attr)
            xfeat_ok = True
            break
    except Exception:
        pass
if not xfeat_ok:
    raise RuntimeError('No compatible XFeat import path found')

sp_ok = False
for module_name, attr in [
    ('models.superpoint_core', 'SuperPointCore'),
    ('superpoint.superpoint', 'SuperPoint'),
    ('superpoint_pytorch', 'SuperPoint'),
    ('superpoint', 'SuperPoint'),
]:
    try:
        mod = importlib.import_module(module_name)
        if hasattr(mod, attr):
            print('SuperPoint import ok via', module_name, attr)
            sp_ok = True
            break
    except Exception:
        pass
if not sp_ok:
    raise RuntimeError('No compatible SuperPoint import path found')

import train
print('train import ok')
'''

cmd = [sys.executable, '-c', test_script]
print('+', ' '.join(cmd))
subprocess.check_call(cmd, cwd=REPO_DIR, env=env)
print('Preflight import checks passed.')


In [ ]:
#@title 9) Select resume checkpoint (auto-pick if empty)
from pathlib import Path

resume = RESUME_CKPT.strip()
if resume and not Path(resume).exists():
    raise FileNotFoundError(f'RESUME_CKPT does not exist: {resume}')

if not resume:
    ckpt_dir = Path(DRIVE_CKPT_DIR)
    best = ckpt_dir / 'best.pth'
    if best.exists():
        resume = str(best)
    else:
        epochs = sorted(ckpt_dir.glob('epoch_*.pth'))
        if epochs:
            resume = str(epochs[-1])

print('Using resume checkpoint:', resume if resume else '(none)')


In [ ]:
#@title 10) Write explicit training override config
import yaml
from pathlib import Path

base_cfg_path = Path(REPO_DIR) / 'config.yaml'
base_cfg = {}
if base_cfg_path.exists():
    with open(base_cfg_path, 'r') as f:
        base_cfg = yaml.safe_load(f) or {}

override_cfg = dict(base_cfg)
override_cfg.update({
    'mode': 'megadepth_raw',
    'data_root': RAW_ROOT,
    'checkpoint_dir': DRIVE_CKPT_DIR,
    'log_dir': DRIVE_LOG_DIR,
    'batch_size': int(BATCH_SIZE),
    'max_epochs': int(MAX_EPOCHS),
    'lr': float(LR),
    'num_workers': int(NUM_WORKERS),
    'image_height': int(IMAGE_H),
    'image_width': int(IMAGE_W),
    'megadepth_val_split_ratio': float(VAL_SPLIT_RATIO),
    'max_pairs_per_scene': int(MAX_PAIRS_PER_SCENE),
    'min_overlap': float(MIN_OVERLAP),
    'max_overlap': float(MAX_OVERLAP),
    'min_keypoint_score': float(MIN_KEYPOINT_SCORE),
    'verify_dataset_pairs': (not bool(NO_VERIFY_DATASET_PAIRS)),
})

override_cfg_path = Path(OVERRIDE_CFG_PATH)
override_cfg_path.write_text(yaml.safe_dump(override_cfg, sort_keys=False))
print('Wrote config override to', override_cfg_path)
print(override_cfg_path.read_text())


In [ ]:
#@title 11) Start TensorBoard (live metrics)
%load_ext tensorboard
%tensorboard --logdir $DRIVE_LOG_DIR --host 127.0.0.1 --port 6006


In [ ]:
#@title 12) Launch MegaDepth Raw training (stream logs)
import os
import sys
import subprocess

env = os.environ.copy()
env['PYTHONPATH'] = ':'.join([REPO_DIR, XFEAT_DIR, SUPERPOINT_DIR, env.get('PYTHONPATH', '')]).strip(':')
env['PYTHONUNBUFFERED'] = '1'

cmd = [
    sys.executable, f'{REPO_DIR}/train.py',
    '--config', str(OVERRIDE_CFG_PATH),
]
if NO_VERIFY_DATASET_PAIRS:
    cmd.append('--no_verify_dataset_pairs')
if resume:
    cmd += ['--resume', resume]

print('Running command\n', ' '.join(cmd))
proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='')

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}')

print('Training finished successfully.')


In [ ]:
#@title 13) List saved checkpoints and logs
from pathlib import Path

ckpt_dir = Path(DRIVE_CKPT_DIR)
log_dir = Path(DRIVE_LOG_DIR)
ckpts = sorted(ckpt_dir.glob('*.pth'))

print('Checkpoint dir:', ckpt_dir)
print('Found', len(ckpts), 'checkpoint files')
for p in ckpts[-30:]:
    print('-', p.name)

print('\nTensorBoard log dir:', log_dir)
if log_dir.exists():
    event_files = sorted(log_dir.glob('**/events.out.tfevents.*'))
    print('Event files:', len(event_files))
else:
    print('Log directory does not exist yet.')
